# Live Alerts Pipeline — End-to-End Validation

Runs every alert enabled by the current client configuration against live adapter data. It executes source quality checks, detection, scoring, fact gathering, and narrative generation.

This notebook does **not** write alert cards or modify the production alerts table. Snowpark may create temporary cached results for the session.

## 1. Run settings

Leave `ANALYSIS_DATE` as `None` to use today. Set `STORE_IDS` to a list for a faster smoke test, or leave it as `None` to validate all stores.

In [ ]:
import datetime

CLIENT = 'eastofengland'
ANALYSIS_DATE = None
STORE_IDS = None

analysis_date = ANALYSIS_DATE or str(datetime.date.today())
print(f'Client: {CLIENT}')
print(f'Analysis date: {analysis_date}')
print(f'Stores: {STORE_IDS or "all"}')

## 2. Load the current configuration and connect to Snowflake

In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session
from store_insight.pipeline.pipeline.capabilities import discover_capabilities
from store_insight.pipeline.pipeline.capabilities import validate as validate_capabilities

cfg = load_config(CLIENT)
capabilities = discover_capabilities(cfg.client_name)
capability_issues = validate_capabilities(cfg)
if capability_issues:
    raise RuntimeError('Capability validation failed:\n- ' + '\n- '.join(capability_issues))

print(f'Config loaded: client={cfg.client_name}')
print(f'Capabilities: {sorted(capabilities) or "(none discovered)"}')

In [ ]:
session = create_session(cfg)

## 3. Derive the production date windows and run the source quality gate

In [ ]:
from store_insight.pipeline.pipeline.pipeline import derive_windows
from store_insight.pipeline.stages._0_quality.quality_gate import run_quality_gate

windows = derive_windows(analysis_date, cfg)
for name, value in windows.items():
    print(f'{name:18} {value}')

gate_plan = run_quality_gate(session, cfg, analysis_date=analysis_date)
print('Source quality gate completed.')
print(gate_plan)

## 4. Load live adapter data

In [ ]:
from store_insight.pipeline.pipeline.run_batch import load_feedback, load_osa, load_wmd

osa = load_osa(session, windows['lookback_start'], windows['this_week_end'], cfg=cfg)
feedback = load_feedback(
    session, windows['last_week_start'], windows['this_week_end'],
    cfg=cfg, capabilities=capabilities,
)
wmd = load_wmd(
    session, windows['wmd_lookback'], windows['this_week_end'],
    cfg=cfg, capabilities=capabilities,
)

live_inputs = {'osa': osa, 'feedback': feedback, **wmd}
for name, frame in live_inputs.items():
    print(f'{name:20} {frame.count():,} rows' if frame is not None else f'{name:20} skipped')

## 5. Run the complete pipeline

`return_intermediates=True` exposes raw and enriched signals for validation. No output writer is called.

In [ ]:
import uuid
from store_insight.pipeline.pipeline.pipeline import run_pipeline

cards, raw_signals, enriched_signals = run_pipeline(
    osa,
    last_week_start=windows['last_week_start'],
    last_week_end=windows['last_week_end'],
    this_week_start=windows['this_week_start'],
    this_week_end=windows['this_week_end'],
    lookback_start=windows['lookback_start'],
    store_id=STORE_IDS,
    session=session,
    feedback=feedback,
    markdown_log=wmd['markdown_log'],
    markdown_sales=wmd['markdown_sales'],
    wastage=wmd['wastage'],
    item_hierarchy=wmd['item_hierarchy'],
    store_hierarchy=wmd['store_hierarchy'],
    cfg=cfg,
    return_intermediates=True,
    run_id=str(uuid.uuid4()),
    pipeline_version=os.environ.get('PIPELINE_VERSION'),
    gate_plan=gate_plan,
)
print(f'Raw signals: {len(raw_signals):,}')
print(f'Enriched signals: {len(enriched_signals):,}')
print(f'Cards: {len(cards):,}')

## 6. Validate coverage and card contracts

In [ ]:
from collections import Counter

raw_counts = Counter(signal.get('signal_type', '<missing>') for signal in raw_signals)
enriched_counts = Counter(signal.get('signal_type', '<missing>') for signal in enriched_signals)
card_counts = Counter(card.get('signal_json', {}).get('signal_type', '<missing>') for card in cards)
all_types = sorted(set(raw_counts) | set(enriched_counts) | set(card_counts))
print(f'{"Signal type":42} {"Raw":>8} {"Enriched":>10} {"Cards":>8}')
print('-' * 72)
for signal_type in all_types:
    print(f'{signal_type:42} {raw_counts[signal_type]:8} {enriched_counts[signal_type]:10} {card_counts[signal_type]:8}')

required_card_fields = {'alert_id', 'signal_json', 'store_id', 'issue', 'explanation', 'cost_if_not_fixed', 'recommended_action'}
invalid_cards = [
    {'index': i, 'missing': sorted(required_card_fields - set(card)), 'card': card}
    for i, card in enumerate(cards)
    if required_card_fields - set(card)
]
assert len(raw_signals) == len(enriched_signals), 'Some detected signals were lost during enrichment'
assert not invalid_cards, f'{len(invalid_cards)} cards are missing required fields'
assert all(card.get('issue') and card.get('recommended_action') for card in cards), 'Blank card narratives found'
print('PASS: enrichment retained every selected signal.')
print('PASS: every generated card has the required fields and non-empty narrative.')

## 7. Inspect one generated card per signal type

In [ ]:
from pprint import pprint

examples = {}
for card in cards:
    signal_type = card.get('signal_json', {}).get('signal_type', '<missing>')
    examples.setdefault(signal_type, card)
for signal_type, card in sorted(examples.items()):
    print(f'\n=== {signal_type} ===')
    pprint(card)

## 8. Close the session

Run this after inspection to release cached Snowflake resources.

In [ ]:
session.close()
print('Snowflake session closed.')